# Relevance Judge Benchmark: run the models

Loads the public dataset from Hugging Face ([`minorproject-research/trace-relevance-judge`](https://huggingface.co/datasets/minorproject-research/trace-relevance-judge),
360 abstracts labeled relevant / not relevant for 30 technical research questions) and tests each candidate judge model on it.

**Resumable by design**

- Every model's answers are written to `results/<model>.jsonl` **one row at a time**, so nothing is lost if the notebook or the machine stops.
- Re-running the notebook **skips models that are already fully tested**. A model that was interrupted resumes with only its missing rows. Rows that failed
  (network error, timeout) are retried automatically.
- To test a model again from scratch, add its name to `FORCE_RERUN` in the configuration cell.
- The result files use the same format as `run_judge_benchmark.py`, so both tools can share the `results/` folder.

**You need:** [Ollama](https://ollama.com) running locally with the models pulled (for the local models), `NVIDIA_API_KEY` in `.env` (for the online ones), and
`pip install datasets httpx python-dotenv pandas tqdm matplotlib` (the next cell does this).

Model tags / ids in `models.json` are placeholders: verify them with `ollama list` and build.nvidia.com. Models that are not installed or not configured are
skipped with a clear message instead of failing.

In [ ]:
# Run once (safe to re-run). Works in Jupyter, VS Code and Colab.
%pip install -q datasets httpx python-dotenv pandas tqdm matplotlib

## 1. Configuration

In [ ]:
import json, math, os, re, statistics, time
from datetime import datetime
from pathlib import Path

import httpx
from dotenv import find_dotenv, load_dotenv

# .env is searched upward from the current folder (NVIDIA_API_KEY, optional HF_TOKEN / OLLAMA_URL).
# override=True so the project's .env wins over system-level variables.
load_dotenv(find_dotenv(usecwd=True), override=True)

# ----------------------------------------------------------------------------- settings
HF_DATASET = "minorproject-research/trace-relevance-judge"
SPLIT = "test"
RESULTS_DIR = Path.cwd() / "results"       # <model>.jsonl per model (same format as run_judge_benchmark.py)

LIMIT = None             # evenly spaced subset for a quick smoke test, e.g. 30. None = all rows
RUN_ONLY = None          # e.g. ["qwen3.5-4b-q8", "gpt-oss-20b"]. None = every model below
FORCE_RERUN = []         # model names whose saved results are deleted and tested again

CONF_THRESHOLD = 0.6        # production rule: a paper passes if relevant and confidence >= this
MAX_ABSTRACT_CHARS = 2500   # same as JUDGE_ABSTRACT_CHAR_LIMIT in relevance_filter.py (labels were written against this)
REQUEST_TIMEOUT = 300       # seconds per model call
MAX_CONSECUTIVE_ERRORS = 8  # stop testing a model after this many failed calls in a row

OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
NVIDIA_URL = "https://integrate.api.nvidia.com/v1/chat/completions"

# ----------------------------------------------------------------------------- models
# backend "ollama": "model" is the tag shown by `ollama list`; "think": False disables hidden reasoning (use "low" for gpt-oss).
# backend "nvidia": "model" is the id from build.nvidia.com. All tags / ids below are PLACEHOLDERS: verify them.
DEFAULT_MODELS = [
    {"name": "qwen3.5-4b-q8",  "backend": "ollama", "model": "hf.co/unsloth/Qwen3.5-4B-GGUF:Q8_0", "think": False},
    {"name": "qwen3.5-9b-q8",  "backend": "ollama", "model": "hf.co/unsloth/Qwen3.5-9B-GGUF:Q8_0", "think": False},
    {"name": "qwen3-8b-q8",    "backend": "ollama", "model": "hf.co/unsloth/Qwen3-8B-GGUF:Q8_0", "think": False},
    {"name": "phi4-14b-q4km",  "backend": "ollama", "model": "hf.co/bartowski/phi-4-GGUF:Q4_K_M"},
    {"name": "phi4-mini-q8",   "backend": "ollama", "model": "hf.co/unsloth/Phi-4-mini-instruct-GGUF:Q8_0"},
    {"name": "gemma4-e4b-q8",  "backend": "ollama", "model": "hf.co/unsloth/gemma-4-E4B-it-GGUF:Q8_0"},
    {"name": "gpt-oss-20b",    "backend": "ollama", "model": "gpt-oss:20b", "think": "low"},
    {"name": "glm-5.3",        "backend": "nvidia", "model": "TODO/glm-5.3"},
    {"name": "glm-5.3-flash",  "backend": "nvidia", "model": "TODO/glm-5.3-flash"},
    {"name": "kimi-k3",        "backend": "nvidia", "model": "TODO/kimi-k3"},
    {"name": "nemotron-3.5-lightning-30b", "backend": "nvidia", "model": "nvidia/nemotron-3.5-lightning-30b-a3b"},
    {"name": "nemotron-3-ultra-550b",      "backend": "nvidia", "model": "nvidia/nemotron-3-ultra-550b-a55b"},
]
# If a models.json sits next to this notebook it is used instead (same format as the repo's models.json).
MODELS_FILE = Path("models.json")
MODELS = json.loads(MODELS_FILE.read_text(encoding="utf-8"))["models"] if MODELS_FILE.exists() else DEFAULT_MODELS

print(f"{len(MODELS)} models configured | results folder: {RESULTS_DIR}")
for m in MODELS:
    print(f"  {m['name']:<28} {m['backend']:<7} {m['model']}")

## 2. Load the dataset from Hugging Face

In [ ]:
from datasets import load_dataset

def load_rows():
    """The dataset is public, so no token is needed; fall back to HF_TOKEN if it is ever made private."""
    try:
        ds = load_dataset(HF_DATASET, split=SPLIT, token=False)
    except Exception:
        token = os.getenv("HF_TOKEN")
        if not token:
            raise
        ds = load_dataset(HF_DATASET, split=SPLIT, token=token)
    return [dict(r) for r in ds]

ROWS = load_rows()
if LIMIT:
    step = max(1, len(ROWS) // LIMIT)
    ROWS = ROWS[::step][:LIMIT]          # evenly spaced, so every sub-question is represented

GOLD = {r["id"]: r for r in ROWS}
ROW_IDS = set(GOLD)

n_pos = sum(r["relevant"] for r in ROWS)
print(f"{len(ROWS)} rows from {HF_DATASET} | {n_pos} relevant ({n_pos / len(ROWS):.0%}) / {len(ROWS) - n_pos} not relevant")

# Reference points every model should beat
majority = max(n_pos, len(ROWS) - n_pos) / len(ROWS)
best_thr_acc, best_thr = max((sum((r["embedding_similarity"] >= t) == r["relevant"] for r in ROWS) / len(ROWS), t / 100)
                             for t in range(40, 95))
BASELINES = {"always 'relevant' (majority class)": majority, f"best embedding-similarity threshold ({best_thr:.2f})": best_thr_acc}
for k, v in BASELINES.items():
    print(f"  baseline accuracy | {k}: {v:.3f}")

## 3. The judge prompt and answer parsing

The prompt is a copy of `RELEVANCE_JUDGE_PROMPT` from `apps/api/agents/stage2_retrieval/relevance_filter.py`, so the notebook measures exactly what runs in
production. If the production prompt changes, update it here as well.

In [ ]:
JUDGE_PROMPT = """You are judging whether a research paper is relevant to a specific research question, based only on its title and abstract.

Research question: {sub_question}

Paper title: {title}
Paper abstract: {abstract}

Judge if this paper is directly relevant and useful as evidence for answering the research question.
Return ONLY a JSON object, nothing else, in this exact format:
{{"relevant": true or false, "confidence": a number between 0 and 1, "reason": "one short sentence"}}
"""

def build_prompt(row):
    return JUDGE_PROMPT.format(sub_question=row["sub_question"], title=row["title"],
                               abstract=(row["abstract"] or "")[:MAX_ABSTRACT_CHARS])

def parse_verdict(raw):
    """Pull the verdict JSON out of a reply (tolerates <think> blocks, code fences, extra prose). None if unusable."""
    if not raw:
        return None
    text = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
    candidates = [text]
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        candidates.append(m.group(0))
    for c in candidates:
        c = c.strip().strip("`")
        if c.startswith("json"):
            c = c[4:]
        try:
            obj = json.loads(c)
        except Exception:
            continue
        if isinstance(obj, dict) and isinstance(obj.get("relevant"), bool):
            try:
                conf = float(obj.get("confidence", 0.0))
            except (TypeError, ValueError):
                conf = 0.0
            return {"relevant": obj["relevant"], "confidence": min(max(conf, 0.0), 1.0), "reason": str(obj.get("reason", ""))}
    return None

print(build_prompt(ROWS[0])[:700], "...")

## 4. Model backends and readiness checks

Local models are called through Ollama, online models through the NVIDIA OpenAI-compatible endpoint. Each Ollama model is unloaded from memory as soon as its test ends, so the next one does not have to share the GPU with it. Before a model is run the notebook checks that it can
actually be used (Ollama reachable and the model pulled, NVIDIA key present and the id not a `TODO`); otherwise it is skipped with the reason.

In [ ]:
def call_ollama(client, cfg, prompt):
    body = {"model": cfg["model"], "messages": [{"role": "user", "content": prompt}], "stream": False,
            "format": "json", "options": {"temperature": 0.1}, "keep_alive": "30m"}
    if "think" in cfg:
        body["think"] = cfg["think"]
    r = client.post(f"{OLLAMA_URL}/api/chat", json=body)
    r.raise_for_status()
    return r.json()["message"]["content"]

def call_nvidia(client, cfg, prompt):
    headers = {"Authorization": f"Bearer {os.environ['NVIDIA_API_KEY']}"}
    body = {"model": cfg["model"], "temperature": 0.1, "max_tokens": 1024, "messages": [{"role": "user", "content": prompt}]}
    for attempt in range(5):
        r = client.post(NVIDIA_URL, headers=headers, json=body)
        if r.status_code in (429, 502, 503):          # rate limit / transient: back off and retry
            time.sleep(2 ** attempt * 3)
            continue
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"] or ""
    raise RuntimeError("NVIDIA endpoint kept rate-limiting")

BACKENDS = {"ollama": call_ollama, "nvidia": call_nvidia}

_installed = {}
def ollama_models():
    """Names of the models Ollama has installed, or None if the server is not reachable."""
    if "names" not in _installed:
        try:
            r = httpx.get(f"{OLLAMA_URL}/api/tags", timeout=5)
            r.raise_for_status()
            _installed["names"] = {x.get(k) for x in r.json().get("models", []) for k in ("name", "model")} - {None}
        except Exception:
            _installed["names"] = None
    return _installed["names"]

def model_ready(cfg):
    """(ok, reason). Lets the run skip a model that cannot work instead of producing hundreds of errors."""
    if cfg["backend"] == "ollama":
        names = ollama_models()
        if names is None:
            return False, f"Ollama is not reachable at {OLLAMA_URL} (start it with `ollama serve`)"
        tag = cfg["model"]
        wanted = {tag, tag if ":" in tag.split("/")[-1] else tag + ":latest"}
        if not wanted & names:
            return False, f"'{tag}' is not installed in Ollama (see `ollama list`; install with `ollama pull {tag}`)"
        return True, ""
    if cfg["backend"] == "nvidia":
        if "TODO" in cfg["model"]:
            return False, f"model id '{cfg['model']}' is still a TODO placeholder (set the real id from build.nvidia.com)"
        if not os.getenv("NVIDIA_API_KEY"):
            return False, "NVIDIA_API_KEY is not set in .env"
        return True, ""
    return False, f"unknown backend '{cfg['backend']}'"

def warm_up(cfg):
    """Load an Ollama model into memory first so the first real row does not carry the load time in its latency."""
    if cfg["backend"] != "ollama":
        return
    try:
        with httpx.Client(timeout=REQUEST_TIMEOUT) as c:
            call_ollama(c, cfg, "Reply with the single word: ok")
    except Exception:
        pass

def unload(cfg):
    """Free the model's memory in Ollama so the next (possibly large) model does not have to share the GPU with it."""
    if cfg["backend"] != "ollama":
        return
    try:
        httpx.post(f"{OLLAMA_URL}/api/generate", json={"model": cfg["model"], "keep_alive": 0}, timeout=30)
    except Exception:
        pass

## 5. Saving results and resuming

`load_done` reads a model's saved rows. A row that carries an error, and any half-written line left by a crash, is dropped so it gets retried. A model counts
as **tested** when every row of the dataset has a saved answer.

In [ ]:
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw):
        return it

def result_path(name):  return RESULTS_DIR / f"{name}.jsonl"
def summary_path(name): return RESULTS_DIR / f"{name}.summary.json"

def load_done(name):
    """{row id: saved result} for a model. Error rows / corrupt lines are removed from the file so they are retried."""
    path = result_path(name)
    if not path.exists():
        return {}
    good, dropped = {}, False
    with path.open(encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:          # half-written last line after a crash
                dropped = True
                continue
            if rec.get("error") or "id" not in rec:
                dropped = True
                continue
            good[rec["id"]] = rec
    if dropped:                                    # rewrite the file cleanly (atomic replace)
        tmp = path.with_suffix(".tmp")
        tmp.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in good.values()), encoding="utf-8")
        tmp.replace(path)
    return good

def is_complete(name):
    return ROW_IDS <= set(load_done(name))

def run_model(cfg):
    """Test one model on the rows it has no saved answer for. Each answer is written to disk immediately."""
    name = cfg["name"]
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    todo = [r for r in ROWS if r["id"] not in load_done(name)]
    call = BACKENDS[cfg["backend"]]
    errors = consecutive = 0
    last_error = ""
    with httpx.Client(timeout=REQUEST_TIMEOUT) as client, result_path(name).open("a", encoding="utf-8") as f:
        for row in tqdm(todo, desc=name):
            t0 = time.perf_counter()
            try:
                raw = call(client, cfg, build_prompt(row))
            except Exception as e:                  # network error, timeout, HTTP error: retried on the next run
                errors += 1
                consecutive += 1
                last_error = f"{type(e).__name__}: {str(e)[:200]}"
                if consecutive >= MAX_CONSECUTIVE_ERRORS:
                    print(f"  stopping {name}: {consecutive} failed calls in a row (last: {last_error})")
                    break
                continue
            consecutive = 0
            rec = {"id": row["id"], "verdict": parse_verdict(raw), "raw": raw[:1500], "error": None,
                   "latency_s": round(time.perf_counter() - t0, 3)}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            f.flush()
    return is_complete(name), errors, last_error

## 6. Scoring

Positive class = *relevant*. A reply that cannot be parsed counts as "not relevant", as it does in production. `hard_*` is the middle band where the cheap
embedding filter is unsure (embedding similarity 0.6 to 0.8), which is where an LLM judge has to earn its cost.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return 0.0, 0.0
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    m = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - m) / d, (c + m) / d

def prf(pairs):
    """pairs = (gold, predicted)."""
    if not pairs:
        return {"acc": float("nan"), "lo": float("nan"), "hi": float("nan"), "prec": float("nan"), "rec": float("nan"), "f1": float("nan")}
    tp = sum(g and p for g, p in pairs)
    fp = sum((not g) and p for g, p in pairs)
    fn = sum(g and (not p) for g, p in pairs)
    ok = sum(g == p for g, p in pairs)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    lo, hi = wilson(ok, len(pairs))
    return {"acc": ok / len(pairs), "lo": lo, "hi": hi, "prec": prec, "rec": rec, "f1": f1}

def score_model(name):
    done = load_done(name)
    res = {i: done[i] for i in GOLD if i in done}
    if not res:
        return None
    raw_pairs, prod_pairs, hard_pairs, briers, conf_ok, conf_bad = [], [], [], [], [], []
    valid = 0
    for rid, rec in res.items():
        gold = GOLD[rid]
        g, v = gold["relevant"], rec["verdict"]
        if v is None:
            pred, pred_prod = False, False
        else:
            valid += 1
            pred, pred_prod = v["relevant"], v["relevant"] and v["confidence"] >= CONF_THRESHOLD
            p_rel = v["confidence"] if v["relevant"] else 1 - v["confidence"]
            briers.append((p_rel - float(g)) ** 2)
            (conf_ok if pred == g else conf_bad).append(v["confidence"])
        raw_pairs.append((g, pred))
        prod_pairs.append((g, pred_prod))
        if 0.6 <= gold["embedding_similarity"] < 0.8:
            hard_pairs.append((g, pred))
    lat = sorted(r["latency_s"] for r in res.values())
    raw, prod, hard = prf(raw_pairs), prf(prod_pairs), prf(hard_pairs)
    mean = lambda xs: statistics.mean(xs) if xs else float("nan")
    return {
        "model": name, "n": len(res), "complete": ROW_IDS <= set(res), "valid_json": valid / len(res),
        "acc": raw["acc"], "acc_lo": raw["lo"], "acc_hi": raw["hi"], "prec": raw["prec"], "rec": raw["rec"], "f1": raw["f1"],
        "prod_f1": prod["f1"], "hard_acc": hard["acc"], "hard_f1": hard["f1"], "brier": mean(briers),
        "conf_right": mean(conf_ok), "conf_wrong": mean(conf_bad),
        "lat_mean": mean(lat), "lat_p95": lat[int(0.95 * (len(lat) - 1))] if lat else float("nan"),
    }

def save_summary(name):
    s = score_model(name)
    summary_path(name).write_text(json.dumps({**s, "dataset": HF_DATASET, "dataset_rows": len(ROWS),
                                              "conf_threshold": CONF_THRESHOLD, "finished_at": datetime.now().isoformat(timespec="seconds")},
                                             indent=2), encoding="utf-8")
    return s

## 7. Run the models

Models are tested **one after another**. A model that is already fully tested is skipped, an interrupted one resumes, and a model that cannot run
(not installed, no key, placeholder id) is skipped with the reason. You can stop the cell at any time (the progress is kept) and run it again later.

In [ ]:
selected = [m for m in MODELS if RUN_ONLY is None or m["name"] in RUN_ONLY]
unknown = set(RUN_ONLY or []) - {m["name"] for m in MODELS}
if unknown:
    print("RUN_ONLY names not found in MODELS:", sorted(unknown))

for name in FORCE_RERUN:
    for p in (result_path(name), summary_path(name)):
        if p.exists():
            p.unlink()
            print(f"[force] deleted {p.name}")

status = {}
for cfg in selected:
    name = cfg["name"]
    if is_complete(name):
        print(f"[skip] {name}: already tested ({len(ROWS)} rows)")
        status[name] = "already tested"
        continue
    ok, why = model_ready(cfg)
    if not ok:
        print(f"[skip] {name}: {why}")
        status[name] = f"not run: {why}"
        continue
    have = len([i for i in load_done(name) if i in ROW_IDS])
    print(f"[run]  {name}: {have}/{len(ROWS)} rows already saved" + (", resuming" if have else ""))
    try:
        warm_up(cfg)
        complete, errors, last_error = run_model(cfg)
    except KeyboardInterrupt:
        print(f"[stop] interrupted during {name}; progress is saved, run this cell again to continue")
        status[name] = "interrupted"
        break
    finally:
        unload(cfg)          # release the model's memory before the next one loads (also after an interrupt or error)
    if complete:
        s = save_summary(name)
        print(f"[done] {name}: acc {s['acc']:.3f}  F1 {s['f1']:.3f}  valid JSON {s['valid_json']:.0%}  latency {s['lat_mean']:.2f}s")
        status[name] = "tested"
    else:
        print(f"[warn] {name}: incomplete ({errors} failed calls; last error: {last_error}). Run this cell again to retry the missing rows.")
        status[name] = "incomplete"

print("\nstatus:")
for k, v in status.items():
    print(f"  {k:<28} {v}")

## 8. Results

Compare every model against the two baselines printed in section 2. With a few hundred rows, accuracy differences smaller than about 5 points are within noise.

In [ ]:
import pandas as pd

scores = [s for s in (score_model(m["name"]) for m in MODELS) if s]
if not scores:
    print("No results yet. Run section 7 first.")
else:
    df = pd.DataFrame(scores).sort_values(["complete", "f1"], ascending=False).reset_index(drop=True)
    show = pd.DataFrame({
        "model": df["model"], "rows": df["n"], "tested": df["complete"].map({True: "yes", False: "partial"}),
        "valid JSON": df["valid_json"].map("{:.0%}".format),
        "accuracy": df["acc"].round(3), "95% CI": [f"{a:.2f}-{b:.2f}" for a, b in zip(df["acc_lo"], df["acc_hi"])],
        "precision": df["prec"].round(3), "recall": df["rec"].round(3), "F1": df["f1"].round(3),
        f"F1 @conf>={CONF_THRESHOLD}": df["prod_f1"].round(3),
        "hard-band acc": df["hard_acc"].round(3), "hard-band F1": df["hard_f1"].round(3),
        "Brier": df["brier"].round(3), "conf right": df["conf_right"].round(2), "conf wrong": df["conf_wrong"].round(2),
        "latency s": df["lat_mean"].round(2), "p95 s": df["lat_p95"].round(2),
    })
    display(show)
    print("baseline accuracy:", {k: round(v, 3) for k, v in BASELINES.items()})

    def to_markdown(t):
        head = "| " + " | ".join(map(str, t.columns)) + " |\n|" + "|".join("---" for _ in t.columns) + "|\n"
        return head + "\n".join("| " + " | ".join(map(str, r)) + " |" for r in t.itertuples(index=False))

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    show.to_csv(RESULTS_DIR / "summary.csv", index=False)
    (RESULTS_DIR / "summary.md").write_text(
        f"Dataset: {HF_DATASET} ({len(ROWS)} rows). Baseline accuracy: "
        + ", ".join(f"{k} {v:.3f}" for k, v in BASELINES.items()) + "\n\n" + to_markdown(show) + "\n", encoding="utf-8")
    print(f"saved {RESULTS_DIR / 'summary.csv'} and summary.md")

In [ ]:
# Accuracy and F1 per model against the baselines (skipped if matplotlib is missing)
try:
    import matplotlib.pyplot as plt
    if scores:
        d = df.iloc[::-1]
        fig, ax = plt.subplots(figsize=(8, 0.45 * len(d) + 1.5))
        pos = range(len(d))
        ax.barh([p + 0.2 for p in pos], d["acc"], height=0.4, label="accuracy")
        ax.barh([p - 0.2 for p in pos], d["f1"], height=0.4, label="F1 (relevant class)")
        ax.set_yticks(list(pos))
        ax.set_yticklabels(d["model"])
        for (label, v), style in zip(BASELINES.items(), (":", "--")):
            ax.axvline(v, color="gray", linestyle=style, linewidth=1, label=label)
        ax.set_xlim(0, 1)
        ax.legend(loc="lower right", fontsize=8)
        ax.set_title("Relevance judge benchmark")
        plt.tight_layout()
        plt.show()
except ImportError:
    print("matplotlib is not installed, skipping the chart")

## 9. Where does the best model disagree with the labels?

Useful for deciding whether an error is the model's fault or a questionable label (low `label_confidence` rows are the contestable ones).

In [ ]:
complete_models = [s for s in scores if s["complete"]] if scores else []
if not complete_models:
    print("No fully tested model yet.")
else:
    best = max(complete_models, key=lambda s: s["f1"])["model"]
    done = load_done(best)
    wrong = []
    for rid, row in GOLD.items():
        v = done[rid]["verdict"]
        pred = bool(v and v["relevant"])          # an unparseable answer counts as "not relevant"
        if pred != row["relevant"]:
            wrong.append((row, v))
    print(f"{best}: {len(wrong)} of {len(GOLD)} rows disagree with the labels (showing up to 15)\n")
    for row, v in wrong[:15]:
        print(f"[{row['id']}] gold={'relevant' if row['relevant'] else 'not relevant'} (label confidence {row['label_confidence']:.2f}) | "
              f"model={'no valid answer' if v is None else ('relevant' if v['relevant'] else 'not relevant')}"
              + ("" if v is None else f" ({v['confidence']:.2f})"))
        print(f"   question: {row['sub_question']}")
        print(f"   title:    {row['title']}")
        print(f"   gold:     {row['label_reason']}")
        if v:
            print(f"   model:    {v['reason']}")
        print()